In [1]:
import pandas as pd
from Levenshtein import distance
import numpy as np
import re

In [3]:
df_csv = pd.read_excel('2025_01_27_Liste der Entschädigungsämter mit Adressen, Varianten und Archiven.xlsx')
df_csv.head()

,Layout class,Filename,Bundesland,CompensationOffice,Zuständiges Archiv,Zuständiges Archiv - GND-Nummer,Unnamed: 6
0,ABC-Karten,1930_12_13_41_0,Niedersachsen,Stadt Celle,Niedersächsisches Landesarchiv,6510041-4,NaN
1,ABC-Karten,1901_03_02_64_0,Niedersachsen,Kreissonderhilfsausschuß \nDelmenhorst,Niedersächsisches Landesarchiv,6510041-4,NaN
2,ABC-Karten,1907_12_31_115_0,Niedersachsen,Goslar,Niedersächsisches Landesarchiv,6510041-4,NaN
3,ABC-Karten,1909_01_06_66_0,Niedersachsen,Goslar. Stadt,Niedersächsisches Landesarchiv,6510041-4,NaN
4,ABC-Karten,1904_01_08_54_0,Niedersachsen,Göttingen-Stadt,Niedersächsisches Landesarchiv,6510041-4,NaN


In [15]:
df_json = pd.read_json('compoff_extracted.jsonl', lines=True)
df_json.head()

,CompensationOffice1,BZKNr,filename
0,Regierungsbezirksamt für Wiedergutmachung und ...,65477,1875_00_00_101_0.jpg
1,Landesamt für die Wiedergutmachung Karlsruhe,EK 20093 A,1875_04_05_16_0.jpg
2,Niedersachsen,1 27574 c,1875_06_15_14_0.jpg
3,Bezirksamt für Wiedergutmachung Neustadt a.d.W.,200 488,1875_04_15_1_0.jpg
4,Entschädigungsamt Berlin,70 002,1875_07_12_9_0.jpg


In [16]:
print(len(df_csv))
print(len(df_json))

n = df_json['CompensationOffice1'].isin(df_csv['CompensationOffice']).sum()
print(f'Number of matches: {n}')

424
1901284
Number of matches: 1121989


# Matching based on Lvenshtein Distance

In [17]:
compensation_offices = list(df_csv['CompensationOffice'])

# remove linebreaks and sequences of whitespaces
def remove_linebreak(s):
    s = str(s).replace('\n', ' ').replace('\r', ' ')
    return re.sub(r'\s+', ' ', s).strip().lower()

# return the minimum distance and the best match up to a threshold of half the length of the cleaned name
def get_min_distance_dynamic_normalized(name):
    name_clean = remove_linebreak(name)
    threshold = len(name_clean) // 2
    
    min_dist = float('inf')
    best_match = None
    for office in compensation_offices:
        office_clean = remove_linebreak(office)
        d = distance(name_clean, office_clean)
        if d == 0:
            return 0, office
        if d < min_dist:
            min_dist = d
            best_match = office
    if min_dist <= threshold:
        return min_dist, best_match
    return np.nan, None

results = df_json['CompensationOffice1'].apply(get_min_distance_dynamic_normalized)

df_matches = pd.DataFrame({
    'CompensationOffice1': df_json['CompensationOffice1'].apply(remove_linebreak),
    'MatchedOffice': results.apply(lambda x: x[1]),
    'MatchDistance': results.apply(lambda x: x[0])
})


df_matches.head(20)

,CompensationOffice1,MatchedOffice,MatchDistance
0,regierungsbezirksamt für wiedergutmachung und ...,Regierungsbezirksamt \nfür Wiedergutmachung \n...,0.0
1,landesamt für die wiedergutmachung karlsruhe,Landesamt für die Wiedergutmachung \nKarlsruhe,0.0
2,niedersachsen,Niedersachsen,0.0
3,bezirksamt für wiedergutmachung neustadt a.d.w.,Bezirksamt \nfür Wiedergutmachung \nNeustadt a...,2.0
4,entschädigungsamt berlin,Entschädigungsamt Berlin,0.0
5,entschädigungsamt berlin,Entschädigungsamt Berlin,0.0
6,köln,Köln,0.0
7,amt für wiedergutmachung des selbstkantkreises...,Amt für Wiedergutmachung des Selfkantkreises G...,9.0
8,landesentschädigungsamt schleswig-holstein in ...,Landesentschädigungsamt\nSchleswig-Holstein in...,0.0
9,hessen kassel,Hessen Kassel,0.0


In [6]:
print(f"Max MatchDistance: {df_matches['MatchDistance'].max()}")

Max MatchDistance: 67.0


In [7]:
total = len(df_matches)
max_dist = int(df_matches['MatchDistance'].max()) + 1
for i in range(max_dist):
    count = (df_matches['MatchDistance'] == i).sum()

    pct = count / total * 100    
    print(f'MatchDistance = {i}: {count} ({pct:.2f}%)')

MatchDistance = 0: 1422538 (74.82%)
MatchDistance = 1: 64054 (3.37%)
MatchDistance = 2: 24334 (1.28%)
MatchDistance = 3: 22113 (1.16%)
MatchDistance = 4: 43303 (2.28%)
MatchDistance = 5: 10644 (0.56%)
MatchDistance = 6: 24848 (1.31%)
MatchDistance = 7: 6033 (0.32%)
MatchDistance = 8: 5534 (0.29%)
MatchDistance = 9: 21285 (1.12%)
MatchDistance = 10: 4700 (0.25%)
MatchDistance = 11: 4947 (0.26%)
MatchDistance = 12: 10438 (0.55%)
MatchDistance = 13: 3892 (0.20%)
MatchDistance = 14: 2818 (0.15%)
MatchDistance = 15: 2491 (0.13%)
MatchDistance = 16: 2054 (0.11%)
MatchDistance = 17: 2153 (0.11%)
MatchDistance = 18: 2218 (0.12%)
MatchDistance = 19: 5044 (0.27%)
MatchDistance = 20: 1473 (0.08%)
MatchDistance = 21: 1185 (0.06%)
MatchDistance = 22: 13153 (0.69%)
MatchDistance = 23: 1518 (0.08%)
MatchDistance = 24: 17725 (0.93%)
MatchDistance = 25: 582 (0.03%)
MatchDistance = 26: 1000 (0.05%)
MatchDistance = 27: 269 (0.01%)
MatchDistance = 28: 242 (0.01%)
MatchDistance = 29: 635 (0.03%)
MatchDista

In [8]:
df_nomatches = df_matches[(df_matches['MatchDistance'] > 0) | (df_matches['MatchDistance'].isna())]

df_nomatches.sort_values('MatchDistance', ascending=True).to_excel('compensation_office_nomatches.xlsx', index=False)

In [18]:
df_exact_matches_or_nan = df_matches[(df_matches['MatchDistance'] == 0) | (df_matches['CompensationOffice1'].isna()) | (df_matches['CompensationOffice1'] == 'nan')]

df_exact_matches_or_nan

,CompensationOffice1,MatchedOffice,MatchDistance
0,regierungsbezirksamt für wiedergutmachung und ...,Regierungsbezirksamt \nfür Wiedergutmachung \n...,0.0
1,landesamt für die wiedergutmachung karlsruhe,Landesamt für die Wiedergutmachung \nKarlsruhe,0.0
2,niedersachsen,Niedersachsen,0.0
4,entschädigungsamt berlin,Entschädigungsamt Berlin,0.0
5,entschädigungsamt berlin,Entschädigungsamt Berlin,0.0
...,...,...,...
1901279,bayerisches landesentschädigungsamt,Bayerisches Landesentschädigungsamt,0.0
1901280,entschädigungsamt berlin,Entschädigungsamt Berlin,0.0
1901281,entschädigungsamt berlin,Entschädigungsamt Berlin,0.0
1901282,bezirksamt für wiedergutmachung koblenz,Bezirksamt für Wiedergutmachung Koblenz,0.0


In [26]:
df_json["norm_office"] = df_json["CompensationOffice1"].apply(remove_linebreak)

exact_norm = set(df_exact_matches_or_nan["CompensationOffice1"].dropna())

df_json_filtered = df_json[~df_json["norm_office"].isin(exact_norm)].copy()

print(len(df_json), len(df_json_filtered))
df_json_filtered.head()

1901284 403408


,CompensationOffice1,BZKNr,filename,norm_office
3,Bezirksamt für Wiedergutmachung Neustadt a.d.W.,200 488,1875_04_15_1_0.jpg,bezirksamt für wiedergutmachung neustadt a.d.w.
7,Amt für Wiedergutmachung des Selbstkantkreises...,48 583,1875_12_26_5_0.jpg,amt für wiedergutmachung des selbstkantkreises...
12,Bezirksamt für Wiedergutmachung Köln,407384,1875_00_00_11_0.jpg,bezirksamt für wiedergutmachung köln
13,Der Regierungspräsidium Düsseldorf,78872,1875_06_17_4_0.jpg,der regierungspräsidium düsseldorf
16,"Hansestadt Hamburg, Sozialbehörde, Amt für Wie...",21806 Keb,1875_09_14_7_0.jpg,"hansestadt hamburg, sozialbehörde, amt für wie..."


In [27]:
df_json_filtered.to_json('compoff_extracted_filtered.jsonl', orient='records', lines=True)

# Continuing with Partial Dataset

In [5]:
df_json_filtered = pd.read_json('compoff_extracted_filtered.jsonl', lines=True)

df_json_filtered

,CompensationOffice1,BZKNr,filename,norm_office
0,Bezirksamt für Wiedergutmachung Neustadt a.d.W.,200 488,1875_04_15_1_0.jpg,bezirksamt für wiedergutmachung neustadt a.d.w.
1,Amt für Wiedergutmachung des Selbstkantkreises...,48 583,1875_12_26_5_0.jpg,amt für wiedergutmachung des selbstkantkreises...
2,Bezirksamt für Wiedergutmachung Köln,407384,1875_00_00_11_0.jpg,bezirksamt für wiedergutmachung köln
3,Der Regierungspräsidium Düsseldorf,78872,1875_06_17_4_0.jpg,der regierungspräsidium düsseldorf
4,"Hansestadt Hamburg, Sozialbehörde, Amt für Wie...",21806 Keb,1875_09_14_7_0.jpg,"hansestadt hamburg, sozialbehörde, amt für wie..."
...,...,...,...,...
403403,Aachen,41 952,1917_03_27_67_0.jpg,aachen
403404,Bayrisches Landesamt für Verfolgtenwesen,123312/VIII/55,1917_06_05_91_0.jpg,bayrisches landesamt für verfolgtenwesen
403405,Reg.-Präsident / Präsident des Verw.Bezirks,1 32601 a,1917_05_24_21_0.jpg,reg.-präsident / präsident des verw.bezirks
403406,Bezirksamt für Wiedergutmachung,111 427,1917_03_27_4_0.jpg,bezirksamt für wiedergutmachung


# Matching without Punctuation

In [8]:
def remove_linebreak(s):
    s = str(s).replace('\n', ' ').replace('\r', ' ')
    return re.sub(r'\s+', ' ', s).strip().lower()

# remove linebreaks, punctuation and sequences of whitespaces
def remove_punctuation(s):
    p = ['?', '!', '.', ',', ';', ':', '-', '_', '(', ')', '[', ']', '{', '}', '"', "'", '/']
    for char in p:
        s = s.replace(char, ' ')
    return re.sub(r'\s+', ' ', s).strip().lower()

compensation_offices = list(df_csv['CompensationOffice'])

print(compensation_offices[:10])

compensation_offices = [remove_linebreak(remove_punctuation(office)) for office in compensation_offices]

print(compensation_offices[:10])

['Stadt Celle', 'Kreissonderhilfsausschuß \nDelmenhorst', 'Goslar', 'Goslar. Stadt', 'Göttingen-Stadt', 'Hannover', 'Hannover - Stadt', 'Stadt Hannover', 'Helmstedt', 'Hildesheim-Stadt']
['stadt celle', 'kreissonderhilfsausschuß delmenhorst', 'goslar', 'goslar stadt', 'göttingen stadt', 'hannover', 'hannover stadt', 'stadt hannover', 'helmstedt', 'hildesheim stadt']


In [9]:

#updated version of get_min_distance_dynamic_normalized that also removes punctuation
def get_min_distance_dynamic_normalized(name):
    name_clean = remove_linebreak(remove_punctuation(name))
    threshold = len(name_clean) // 2
    
    min_dist = float('inf')
    best_match = None
    for office in compensation_offices:
        d = distance(name_clean, office)
        if d == 0:
            return 0, office
        if d < min_dist:
            min_dist = d
            best_match = office
    if min_dist <= threshold:
        return min_dist, best_match
    return np.nan, None

results = df_json_filtered['CompensationOffice1'].apply(get_min_distance_dynamic_normalized)

df_matches = pd.DataFrame({
    'CompensationOffice1': df_json_filtered['CompensationOffice1'].apply(remove_punctuation),
    'MatchedOffice': results.apply(lambda x: x[1]),
    'MatchDistance': results.apply(lambda x: x[0])
})

df_matches.head(20)

,CompensationOffice1,MatchedOffice,MatchDistance
0,bezirksamt für wiedergutmachung neustadt a d w,bezirksamt für wiedergutmachung neustadt a d w,0.0
1,amt für wiedergutmachung des selbstkantkreises...,amt für wiedergutmachung des selfkantkreises g...,7.0
2,bezirksamt für wiedergutmachung köln,bezirksamt für wiedergutmachung mainz,4.0
3,der regierungspräsidium düsseldorf,der regierungspräsident düsseldorf,3.0
4,hansestadt hamburg sozialbehörde amt für wiede...,hansestadt hamburg sozialbehörde amt für wiede...,1.0
5,regierungsbezirk für wiedergutmachung und verw...,regierungsbezirksamt für wiedergutmachung und ...,4.0
6,hess staatsministerium,NaN,NaN
7,regierungsbezirksamt für wiedergutmachung und ...,regierungsbezirksamt für wiedergutmachung und ...,2.0
8,freie und hansestadt hamburg,NaN,NaN
9,hess staatsministerium,NaN,NaN


In [10]:
total = len(df_matches)
max_dist = int(df_matches['MatchDistance'].max()) + 1
for i in range(max_dist):
    count = (df_matches['MatchDistance'] == i).sum()

    pct = count / total * 100    
    print(f'MatchDistance = {i}: {count} ({pct:.2f}%)')

MatchDistance = 0: 52227 (12.95%)
MatchDistance = 1: 33750 (8.37%)
MatchDistance = 2: 10717 (2.66%)
MatchDistance = 3: 19463 (4.82%)
MatchDistance = 4: 43260 (10.72%)
MatchDistance = 5: 9386 (2.33%)
MatchDistance = 6: 23139 (5.74%)
MatchDistance = 7: 5620 (1.39%)
MatchDistance = 8: 6111 (1.51%)
MatchDistance = 9: 22174 (5.50%)
MatchDistance = 10: 3724 (0.92%)
MatchDistance = 11: 3588 (0.89%)
MatchDistance = 12: 10586 (2.62%)
MatchDistance = 13: 3822 (0.95%)
MatchDistance = 14: 2880 (0.71%)
MatchDistance = 15: 2658 (0.66%)
MatchDistance = 16: 2481 (0.62%)
MatchDistance = 17: 2541 (0.63%)
MatchDistance = 18: 2527 (0.63%)
MatchDistance = 19: 15769 (3.91%)
MatchDistance = 20: 2457 (0.61%)
MatchDistance = 21: 1501 (0.37%)
MatchDistance = 22: 1215 (0.30%)
MatchDistance = 23: 18172 (4.50%)
MatchDistance = 24: 1059 (0.26%)
MatchDistance = 25: 569 (0.14%)
MatchDistance = 26: 233 (0.06%)
MatchDistance = 27: 528 (0.13%)
MatchDistance = 28: 949 (0.24%)
MatchDistance = 29: 1741 (0.43%)
MatchDistanc

In [11]:
df_exact_matches = df_matches[df_matches['MatchDistance'] == 0]

df_exact_matches 

,CompensationOffice1,MatchedOffice,MatchDistance
0,bezirksamt für wiedergutmachung neustadt a d w,bezirksamt für wiedergutmachung neustadt a d w,0.0
11,regierungsbezirksamt für wiedergutmachung und ...,regierungsbezirksamt für wiedergutmachung und ...,0.0
25,regierungsbezirksamt für wiedergutmachung und ...,regierungsbezirksamt für wiedergutmachung und ...,0.0
37,regierungsbezirksamt für wiedergutmachung und ...,regierungsbezirksamt für wiedergutmachung und ...,0.0
39,landesamt für die wiedergutmachung tübingen,landesamt für die wiedergutmachung tübingen,0.0
...,...,...,...
403373,oberfinanzdirektion köln,oberfinanzdirektion köln,0.0
403374,regierungsbezirksamt für wiedergutmachung und ...,regierungsbezirksamt für wiedergutmachung und ...,0.0
403381,bezirksamt für wiedergutmachung neustadt a d w,bezirksamt für wiedergutmachung neustadt a d w,0.0
403391,oberfinanzdirektion köln,oberfinanzdirektion köln,0.0


In [ ]:
df_json_filtered["norm_office"] = df_json_filtered["norm_office"].apply(remove_punctuation)

exact_norm = set(df_exact_matches["CompensationOffice1"].dropna())

df_json_filtered = df_json_filtered[~df_json_filtered["norm_office"].isin(exact_norm)].copy()

df_json_filtered

,CompensationOffice1,BZKNr,filename,norm_office
1,Amt für Wiedergutmachung des Selbstkantkreises...,48 583,1875_12_26_5_0.jpg,amt für wiedergutmachung des selbstkantkreises...
2,Bezirksamt für Wiedergutmachung Köln,407384,1875_00_00_11_0.jpg,bezirksamt für wiedergutmachung köln
3,Der Regierungspräsidium Düsseldorf,78872,1875_06_17_4_0.jpg,der regierungspräsidium düsseldorf
4,"Hansestadt Hamburg, Sozialbehörde, Amt für Wie...",21806 Keb,1875_09_14_7_0.jpg,hansestadt hamburg sozialbehörde amt für wiede...
5,Regierungsbezirk für Wiedergutmachung und verw...,47 691,1875_04_05_13_0.jpg,regierungsbezirk für wiedergutmachung und verw...
...,...,...,...,...
403402,Reg.-Präsident /Präsident des Vereinsbezirks H...,1 27101 a,1917_06_23_32_0.jpg,reg präsident präsident des vereinsbezirks han...
403403,Aachen,41 952,1917_03_27_67_0.jpg,aachen
403404,Bayrisches Landesamt für Verfolgtenwesen,123312/VIII/55,1917_06_05_91_0.jpg,bayrisches landesamt für verfolgtenwesen
403405,Reg.-Präsident / Präsident des Verw.Bezirks,1 32601 a,1917_05_24_21_0.jpg,reg präsident präsident des verw bezirks


In [ ]:
df_no_matches_no_punctuation = df_matches[(df_matches['MatchDistance'] > 0) | (df_matches['MatchDistance'].isna())]
df_no_matches_no_punctuation = df_no_matches_no_punctuation.sort_values('MatchDistance', ascending=False)
df_no_matches_no_punctuation.head(20)

,CompensationOffice1,MatchedOffice,MatchDistance
422246,der regierungspräsident dezernat für wiedergut...,Der Regierungspräsident \n- Dezernat für Wiede...,64.0
47910,freie und hansestadt hamburg behörde für arbei...,"Behörde für Arbeit, Gesunheit und Soziales - ...",61.0
227493,regierungsbezirksamt für wiedergutmachung und ...,Regierungsbezirksamt für Wiedergutmachung und ...,60.0
1135308,der regierungspräsident dezernat für wiedergut...,Der Regierungspräsident \n- Dezernat für Wiede...,60.0
1881670,freie und hansestadt hamburg behörde für arbei...,"Behörde für Arbeit, Gesunheit und Soziales - ...",60.0
1617342,freie und hansestadt hamburg behörde für arbei...,"Behörde für Arbeit, Gesunheit und Soziales - ...",60.0
1138927,der regierungspräsident dezernat für wiedergut...,Der Regierungspräsident \n- Dezernat für Wiede...,60.0
1126699,der regierungspräsident – dezernat für wiederg...,Der Regierungspräsident \n- Dezernat für Wiede...,60.0
293761,freie und hansestadt hamburg behörde für arbei...,"Behörde für Arbeit, Gesunheit und Soziales - ...",60.0
1827229,der regierungspräsident dezernat für wiedergut...,Der Regierungspräsident \n- Dezernat für Wiede...,60.0


In [ ]:
df_no_matches_no_punctuation.to_excel('compensation_office_nomatches_no_punctuation.xlsx', index=False)

# No Matches

In [56]:
no_matched_office_df = df_nomatches[df_nomatches['MatchDistance'].isna()]

no_matched_office_df

,CompensationOffice1,MatchedOffice,MatchDistance
21,hess staatsministerium,NaN,NaN
30,nan,NaN,NaN
32,freie und hansestadt hamburg,NaN,NaN
34,hess staatsministerium,NaN,NaN
40,nan,NaN,NaN
...,...,...,...
1901202,hng,NaN,NaN
1901203,reg präsident präsident des verw bezirks hannover,NaN,NaN
1901238,hng,NaN,NaN
1901254,reg präsident präsident des vereinsbezirks han...,NaN,NaN


In [91]:
no_matched_office_df_set = set(no_matched_office_df['CompensationOffice1'])

len(no_matched_office_df_set)
for office in no_matched_office_df_set:
    print(office)


th bayerisches verfolgungsamt
schweiz
bezirksamt
ansl amtbremen
bmt ub4
landesherzogtum bayern
k817
entschädigungsbehörde landesentschädigungsbehörde für die bundesrepublik deutschland
kreisaußerhilfsausschübe
wiedergutmachung stuttgart
regionalverwaltungsamt neustadt a d weinstr
bmf vb 4
aör amt bremen
beauftragung
sondervermögens u bautverwaltung
hess ns hätefonds
bay staatsarchiv
niedersachsen präsident präsident des vermögensverwaltungsausschusses
kreissozialamt hade
bayerisches volksgericht
kreissozialamt jena
r a sladowsky
regierungsbezirksverwaltungsamt koblenz
land amt mainz
niedersachsen präsident präsident des verwaltungsgerichts aurich
dyepetrouke ukraine
pöltmann annemarie
15 5 23
bayerisches landesamt für die opfer des nationalsozialismus
kupfermints
wuttenstein salzgitter
rey frieda ruth geb ehrmann 8 10 33 kuhlmann jnge geb ehrmann 14 12 35 ehrmann doris renate 31 3 1937
staatsanwalt lg münchen
lka jülich
landsgericht und strafkammer münchen
niedersachsen präsident des 

# Swapping out single Tokens with a match distance of 1

In [16]:
def office_token_set(series):
    tokens = set()
    cleaned = (
        series.fillna("")
        .astype(str)
        .str.lower()
        .str.replace(r"\n", " ", regex=True)
        .str.replace(r"\s+", " ", regex=True)
        .str.strip()
    )
    for name in cleaned:
        tokens.update(token for token in re.findall(r"\b\w+\b", name, flags=re.UNICODE)
            if not token.isdigit())
    return tokens

csv_tokens = office_token_set(df_csv["CompensationOffice"].apply(remove_punctuation).apply(remove_linebreak))
json_tokens = office_token_set(df_json_filtered["norm_office"])

print(f"csv_tokens: {len(csv_tokens)}")
print(f"json_tokens: {len(json_tokens)}")

csv_tokens: 354
json_tokens: 10692


In [11]:
for t in csv_tokens:
    if len(t) == 3:
        print(t)

amt
hdt
ems
186
459
dez
neu
493
170
unk
für
reg
abt
die
108
und
süd
vb4
ofd
107
159
172
der
das
des
str
bad
fin
089
lrb
127
577
min
116


In [18]:
from collections import defaultdict

csv_tokens_by_len = defaultdict(list)
for token in csv_tokens:
    if len(token) >= 3:
        csv_tokens_by_len[len(token)].append(token)

json_to_csv_distance1_map = {}
for json_token in json_tokens:
    if json_token in csv_tokens:
        continue

    candidate_tokens = (
        csv_tokens_by_len[len(json_token) - 1]
        + csv_tokens_by_len[len(json_token)]
        + csv_tokens_by_len[len(json_token) + 1]
    )

    matches_dist1 = [csv_token for csv_token in candidate_tokens if distance(json_token, csv_token) == 1]
    if matches_dist1:
        json_to_csv_distance1_map[json_token] = sorted(set(matches_dist1))

single_token_matches_dist1_len3 =sorted(
    [item for item in json_to_csv_distance1_map.items() if len(item[1]) == 1],
    key=lambda item: len(item[0])
)

print(len(single_token_matches_dist1_len3))
single_token_matches_dist1_len3

966


[('re', ['reg']),
 ('st', ['str']),
 ('rb', ['lrb']),
 ('tr', ['str']),
 ('rg', ['reg']),
 ('sr', ['str']),
 ('vb', ['vb4']),
 ('of', ['ofd']),
 ('er', ['der']),
 ('ad', ['bad']),
 ('di', ['die']),
 ('bd', ['bad']),
 ('ba', ['bad']),
 ('uk', ['unk']),
 ('ht', ['hdt']),
 ('eg', ['reg']),
 ('mi', ['min']),
 ('dt', ['hdt']),
 ('da', ['das']),
 ('b4', ['vb4']),
 ('mt', ['amt']),
 ('lr', ['lrb']),
 ('dr', ['der']),
 ('hd', ['hdt']),
 ('nu', ['neu']),
 ('fr', ['für']),
 ('as', ['das']),
 ('dz', ['dez']),
 ('hdm', ['hdt']),
 ('abh', ['abt']),
 ('vr4', ['vb4']),
 ('keg', ['reg']),
 ('baf', ['bad']),
 ('rep', ['reg']),
 ('tür', ['für']),
 ('vd4', ['vb4']),
 ('oft', ['ofd']),
 ('dhe', ['die']),
 ('for', ['für']),
 ('efd', ['ofd']),
 ('mit', ['min']),
 ('vbh', ['vb4']),
 ('ofg', ['ofd']),
 ('per', ['der']),
 ('ub4', ['vb4']),
 ('wür', ['für']),
 ('db4', ['vb4']),
 ('and', ['und']),
 ('beg', ['reg']),
 ('lr8', ['lrb']),
 ('stg', ['str']),
 ('new', ['neu']),
 ('bau', ['bad']),
 ('fln', ['fin']),
 (

In [19]:
unique_distance1_token_map = {
    json_token: csv_candidates[0]
    for json_token, csv_candidates in json_to_csv_distance1_map.items()
    if len(csv_candidates) == 1
}

def replace_unique_distance1_tokens(text):
    text = "" if pd.isna(text) else str(text)
    return re.sub(
        r"\b\w+\b",
        lambda match: unique_distance1_token_map.get(match.group(0).lower(), match.group(0)),
        text,
        flags=re.UNICODE
    )

df_json_filtered['CompensationOffice_token_swap'] = df_json_filtered['norm_office'].apply(replace_unique_distance1_tokens)

print(f"Unique token replacements available: {len(unique_distance1_token_map)}")

results = df_json_filtered['CompensationOffice_token_swap'].apply(get_min_distance_dynamic_normalized)
df_matches_token_swap = pd.DataFrame({
    'CompensationOffice1': df_json_filtered['norm_office'],
    'MatchedOffice': results.apply(lambda x: x[1]),
    'MatchDistance': results.apply(lambda x: x[0])
})

df_matches_token_swap.head(20)

Unique token replacements available: 966


,CompensationOffice1,MatchedOffice,MatchDistance
1,amt für wiedergutmachung des selbstkantkreises...,amt für wiedergutmachung des selfkantkreises g...,7.0
2,bezirksamt für wiedergutmachung köln,bezirksamt für wiedergutmachung mainz,4.0
3,der regierungspräsidium düsseldorf,der regierungspräsident düsseldorf,3.0
4,hansestadt hamburg sozialbehörde amt für wiede...,hansestadt hamburg sozialbehörde amt für wiede...,1.0
5,regierungsbezirk für wiedergutmachung und verw...,regierungsbezirksamt für wiedergutmachung und ...,3.0
6,hess staatsministerium,NaN,NaN
7,regierungsbezirksamt für wiedergutmachung und ...,regierungsbezirksamt für wiedergutmachung und ...,2.0
8,freie und hansestadt hamburg,NaN,NaN
9,hess staatsministerium,NaN,NaN
10,hansestadt hamburg sozialbehörde amt für wiede...,hansestadt hamburg sozialbehörde amt für wiede...,1.0


In [20]:
total = len(df_matches_token_swap)
max_dist = int(df_matches_token_swap['MatchDistance'].max()) + 1
for i in range(max_dist):
    count = (df_matches_token_swap['MatchDistance'] == i).sum()

    pct = count / total * 100    
    print(f'MatchDistance = {i}: {count} ({pct:.2f}%)')

MatchDistance = 0: 12792 (3.64%)
MatchDistance = 1: 21227 (6.04%)
MatchDistance = 2: 11203 (3.19%)
MatchDistance = 3: 20804 (5.92%)
MatchDistance = 4: 41702 (11.87%)
MatchDistance = 5: 9182 (2.61%)
MatchDistance = 6: 23788 (6.77%)
MatchDistance = 7: 4670 (1.33%)
MatchDistance = 8: 6243 (1.78%)
MatchDistance = 9: 22337 (6.36%)
MatchDistance = 10: 3547 (1.01%)
MatchDistance = 11: 3583 (1.02%)
MatchDistance = 12: 10961 (3.12%)
MatchDistance = 13: 3567 (1.02%)
MatchDistance = 14: 2843 (0.81%)
MatchDistance = 15: 2780 (0.79%)
MatchDistance = 16: 2670 (0.76%)
MatchDistance = 17: 2328 (0.66%)
MatchDistance = 18: 2979 (0.85%)
MatchDistance = 19: 15769 (4.49%)
MatchDistance = 20: 3547 (1.01%)
MatchDistance = 21: 1501 (0.43%)
MatchDistance = 22: 1430 (0.41%)
MatchDistance = 23: 18083 (5.15%)
MatchDistance = 24: 1149 (0.33%)
MatchDistance = 25: 539 (0.15%)
MatchDistance = 26: 247 (0.07%)
MatchDistance = 27: 521 (0.15%)
MatchDistance = 28: 993 (0.28%)
MatchDistance = 29: 1717 (0.49%)
MatchDistance

In [21]:
def token_count(s):
    return len(str(s).split())

extracted_longer = sorted(
    {
        (str(a), str(b))
        for a, b in zip(df_matches_token_swap['CompensationOffice1'], df_matches_token_swap['MatchedOffice'])
        if token_count(a) > token_count(b) and str(b) != 'nan'
    },
    key=lambda item: token_count(item[0])
)

extracted_longer

[('hannover köln', 'hannover'),
 ('ksha münchen', 'münchen'),
 ('lg köln', 'köln'),
 ('lg hamburg', 'hamburg'),
 ('neu stadt', 'neustadt'),
 ('bfj berlin', 'berlin'),
 ('führer koblenz', 'koblenz'),
 ('neustadt 484945', 'neustadt'),
 ('köln hamburg', 'hamburg'),
 ('münchen bürgbr', 'münchen'),
 ('köln düsseldorf', 'düsseldorf'),
 ('ksha nordern', 'hannover'),
 ('hannover löd', 'hannover'),
 ('ksh stade', 'stade'),
 ('ovd berlin', 'berlin'),
 ('oldeburg oldb', 'oldenburg'),
 ('oed koblenz', 'koblenz'),
 ('neu neustadt', 'neustadt'),
 ('ofo berlin', 'berlin'),
 ('aegyung aensberg', 'arnsberg'),
 ('nrw tüssleldorf', 'düsseldorf'),
 ('braunschweig mascherode', 'braunschweig'),
 ('neustadt pk', 'neustadt'),
 ('braunschweig ii', 'braunschweig'),
 ('hfa berlin', 'berlin'),
 ('mainz neustadt', 'neustadt'),
 ('tries neustadt', 'neustadt'),
 ('hannover fund', 'hannover'),
 ('stl desheim', 'hildesheim'),
 ('rp düsseldorf', 'düsseldorf'),
 ('münchen beg', 'münchen'),
 ('bayern dt', 'bayern'),
 ('k

# Swapping out single tokens with a match distance of up to half of the token's lenght

In [ ]:
json_to_csv_closest_map = {}
for json_token in json_tokens:
    if json_token in csv_tokens:
        continue
    
    max_distance = len(json_token) // 2
    best_match = None
    best_dist = float('inf')
    
    for csv_token in csv_tokens:
        if len(csv_token) >= 3:
            if abs(len(csv_token) - len(json_token)) > max_distance:
                continue
            dist = distance(json_token, csv_token)
            if dist <= max_distance and dist < best_dist:
                best_dist = dist
                best_match = csv_token
    
    if best_match is not None:
        json_to_csv_closest_map[json_token] = [best_match]

single_token_matches_closest = sorted(
    json_to_csv_closest_map.items(),
    key=lambda item: len(item[0])
)
print(len(single_token_matches_closest))
single_token_matches_closest

6957


[('re', ['reg']),
 ('st', ['str']),
 ('rb', ['lrb']),
 ('tr', ['str']),
 ('rg', ['reg']),
 ('sr', ['str']),
 ('vb', ['vb4']),
 ('of', ['ofd']),
 ('er', ['der']),
 ('ad', ['bad']),
 ('es', ['des']),
 ('di', ['die']),
 ('bd', ['bad']),
 ('ba', ['bad']),
 ('uk', ['unk']),
 ('ht', ['hdt']),
 ('de', ['der']),
 ('eg', ['reg']),
 ('mi', ['min']),
 ('dt', ['hdt']),
 ('da', ['das']),
 ('b4', ['vb4']),
 ('mt', ['amt']),
 ('lr', ['lrb']),
 ('dr', ['der']),
 ('hd', ['hdt']),
 ('nu', ['neu']),
 ('fr', ['für']),
 ('as', ['das']),
 ('dz', ['dez']),
 ('hdm', ['hdt']),
 ('abh', ['abt']),
 ('vr4', ['vb4']),
 ('keg', ['reg']),
 ('ein', ['fin']),
 ('baf', ['bad']),
 ('rep', ['reg']),
 ('tür', ['für']),
 ('vd4', ['vb4']),
 ('oft', ['ofd']),
 ('dhe', ['die']),
 ('for', ['für']),
 ('efd', ['ofd']),
 ('mit', ['min']),
 ('vbh', ['vb4']),
 ('drs', ['das']),
 ('ofg', ['ofd']),
 ('per', ['der']),
 ('ub4', ['vb4']),
 ('alt', ['abt']),
 ('vin', ['fin']),
 ('wür', ['für']),
 ('db4', ['vb4']),
 ('ant', ['abt']),
 ('a

In [30]:
unique_distance1_token_map = {
    json_token: csv_candidates[0]
    for json_token, csv_candidates in json_to_csv_closest_map.items()
    if len(csv_candidates) == 1
}

def replace_unique_distance1_tokens(text):
    text = "" if pd.isna(text) else str(text)
    return re.sub(
        r"\b\w+\b",
        lambda match: unique_distance1_token_map.get(match.group(0).lower(), match.group(0)),
        text,
        flags=re.UNICODE
    )

df_json_filtered['CompensationOffice_token_swap'] = df_json_filtered['norm_office'].apply(replace_unique_distance1_tokens)

print(f"Unique token replacements available: {len(unique_distance1_token_map)}")

results = df_json_filtered['CompensationOffice_token_swap'].apply(get_min_distance_dynamic_normalized)
df_matches_token_swap = pd.DataFrame({
    'CompensationOffice1': df_json_filtered['norm_office'],
    'MatchedOffice': results.apply(lambda x: x[1]),
    'MatchDistance': results.apply(lambda x: x[0])
})

df_matches_token_swap.head(20)

Unique token replacements available: 6957


,CompensationOffice1,MatchedOffice,MatchDistance
1,amt für wiedergutmachung des selbstkantkreises...,amt für wiedergutmachung des selfkantkreises g...,10.0
2,bezirksamt für wiedergutmachung köln,bezirksamt für wiedergutmachung mainz,4.0
3,der regierungspräsidium düsseldorf,der regierungspräsident düsseldorf,3.0
4,hansestadt hamburg sozialbehörde amt für wiede...,hansestadt hamburg sozialbehörde amt für wiede...,1.0
5,regierungsbezirk für wiedergutmachung und verw...,regierungsbezirksamt für wiedergutmachung und ...,3.0
6,hess staatsministerium,NaN,NaN
7,regierungsbezirksamt für wiedergutmachung und ...,regierungsbezirksamt für wiedergutmachung und ...,0.0
8,freie und hansestadt hamburg,NaN,NaN
9,hess staatsministerium,NaN,NaN
10,hansestadt hamburg sozialbehörde amt für wiede...,hansestadt hamburg sozialbehörde amt für wiede...,1.0


In [31]:
total = len(df_matches_token_swap)
max_dist = int(df_matches_token_swap['MatchDistance'].max()) + 1
for i in range(max_dist):
    count = (df_matches_token_swap['MatchDistance'] == i).sum()

    pct = count / total * 100    
    print(f'MatchDistance = {i}: {count} ({pct:.2f}%)')

MatchDistance = 0: 25402 (7.23%)
MatchDistance = 1: 20177 (5.75%)
MatchDistance = 2: 6929 (1.97%)
MatchDistance = 3: 24342 (6.93%)
MatchDistance = 4: 41748 (11.89%)
MatchDistance = 5: 6293 (1.79%)
MatchDistance = 6: 24025 (6.84%)
MatchDistance = 7: 3830 (1.09%)
MatchDistance = 8: 6655 (1.90%)
MatchDistance = 9: 24577 (7.00%)
MatchDistance = 10: 3689 (1.05%)
MatchDistance = 11: 2849 (0.81%)
MatchDistance = 12: 13240 (3.77%)
MatchDistance = 13: 4714 (1.34%)
MatchDistance = 14: 1555 (0.44%)
MatchDistance = 15: 3294 (0.94%)
MatchDistance = 16: 3072 (0.87%)
MatchDistance = 17: 4192 (1.19%)
MatchDistance = 18: 1918 (0.55%)
MatchDistance = 19: 13472 (3.84%)
MatchDistance = 20: 1324 (0.38%)
MatchDistance = 21: 2064 (0.59%)
MatchDistance = 22: 691 (0.20%)
MatchDistance = 23: 17697 (5.04%)
MatchDistance = 24: 1092 (0.31%)
MatchDistance = 25: 433 (0.12%)
MatchDistance = 26: 201 (0.06%)
MatchDistance = 27: 669 (0.19%)
MatchDistance = 28: 1417 (0.40%)
MatchDistance = 29: 1916 (0.55%)
MatchDistance 

# Calling LLM via API using NaN values

In [ ]:
from groq import Groq
from huggingface_hub import InferenceClient
from huggingface_hub.utils import HfHubHTTPError

import time
import json
import os
from dotenv import load_dotenv
from tqdm import tqdm


In [ ]:
load_dotenv()

LLM_PROVIDER = os.getenv("LLM_PROVIDER", "groq").strip().lower()
HF_MODEL = os.getenv("HF_MODEL", "Qwen/Qwen3-8B")
GROQ_MODEL = os.getenv("GROQ_MODEL", "llama-3.1-8b-instant")

if LLM_PROVIDER == "huggingface":
    HF_API_TOKEN = os.getenv("HF_API_KEY")
    if not HF_API_TOKEN:
        raise ValueError("HF_API_KEY is required when LLM_PROVIDER='huggingface'")
    client = InferenceClient(
        model=HF_MODEL,
        token=HF_API_TOKEN
    )
    ACTIVE_MODEL = HF_MODEL
elif LLM_PROVIDER == "groq":
    GROQ_API_KEY = os.getenv("GROQ_API_KEY")
    if not GROQ_API_KEY:
        raise ValueError("GROQ_API_KEY is required when LLM_PROVIDER='groq'")
    client = Groq(api_key=GROQ_API_KEY)
    ACTIVE_MODEL = GROQ_MODEL
else:
    raise ValueError("LLM_PROVIDER must be either 'huggingface' or 'groq'")

print(f"Using LLM provider: {LLM_PROVIDER}, model: {ACTIVE_MODEL}")

offices_list = "\n".join([f"- {office}" for office in compensation_offices])

In [ ]:

def get_llm_match_batch(query_names, offices_list, max_retries=3):
    
    queries = "\n".join([f"{i+1}. \"{name}\"" for i, name in enumerate(query_names)])
    
    prompt = f"""You are an expert at matching German compensation office names (Entschädigungsämter).

Given these query names:
{queries}

Find the BEST match for EACH from this list:
{offices_list}

Rules:
1. Return ONLY a numbered list with the exact name from the list
2. Consider spelling variations and OCR errors
3. If no match exists, write \"NO_MATCH\"
4. Format: \"1. Matching Name\" or \"1. NO_MATCH\"

Answers:"""

    messages = [{"role": "user", "content": prompt}]
    
    for attempt in range(max_retries):
        try:
            if LLM_PROVIDER == "huggingface":
                response = client.chat_completion(
                    messages=messages,
                    max_tokens=500,
                    temperature=0.1
                )
                text = response.choices[0].message.content
            elif LLM_PROVIDER == "groq":
                response = client.chat.completions.create(
                    model=ACTIVE_MODEL,
                    messages=messages,
                    max_tokens=500,
                    temperature=0.1
                )
                text = response.choices[0].message.content
            else:
                raise ValueError(f"Unsupported LLM provider: {LLM_PROVIDER}")
            
            matches = []
            lines = text.split('\n')
            for line in lines:
                line = line.strip()
                if line and len(line) > 0 and line[0].isdigit():
                    parts = line.split('.', 1) if '.' in line else line.split(')', 1)
                    if len(parts) > 1:
                        match = parts[1].strip()
                        matches.append(match)
            
            while len(matches) < len(query_names):
                matches.append("PARSE_ERROR")
            
            return matches[:len(query_names)]
            
        except HfHubHTTPError as e:
            error_str = str(e).lower()
            if "429" in str(e) or "rate" in error_str:
                print(f"Rate limited. Waiting 30s before retry (attempt {attempt+1}/{max_retries})...")
                time.sleep(30)
            elif "402" in str(e) or "quota" in error_str or "exceeded" in error_str:
                print("API quota exceeded! Stopping.")
                return None  # Signal to stop processing
            elif "503" in str(e) or "loading" in error_str:
                print(f"Model loading. Waiting 20s...")
                time.sleep(20)
            else:
                print(f"HTTP Error: {e}")
                if attempt < max_retries - 1:
                    time.sleep(10)
                else:
                    return ["ERROR"] * len(query_names)
        except Exception as e:
            error_str = str(e).lower()
            if "429" in error_str or "rate" in error_str:
                print(f"Rate limited. Waiting 30s before retry (attempt {attempt+1}/{max_retries})...")
                time.sleep(30)
            elif "402" in error_str or "quota" in error_str or "exceeded" in error_str:
                print("API quota exceeded! Stopping.")
                return None  
            elif "503" in error_str or "loading" in error_str:
                print("Model loading. Waiting 20s...")
                time.sleep(20)
            else:
                print(f"Error: {e}")
                if attempt < max_retries - 1:
                    time.sleep(10)
                else:
                    return ["ERROR"] * len(query_names)
    
    return ["ERROR"] * len(query_names)

if len(no_matched_office_df) >= 2:
    test_names = no_matched_office_df['CompensationOffice1'].iloc[:2].tolist()
    print(f"Testing with:")
    for i, name in enumerate(test_names):
        print(f"  {i+1}. {name}")
    print()
    results = get_llm_match_batch(test_names, offices_list)
    if results:
        print("LLM suggested matches:")
        for i, result in enumerate(results):
            print(f"  {i+1}. {result}")
    else:
        print("API limit reached during test!")

Testing with:
  1. hess staatsministerium
  2. nan

HTTP Error: (Request ID: Root=1-69b1eb80-71adc530557736b071097d90;d9f9dd39-1eb3-4ef0-ba3c-5262937b9357)

Bad request:
HTTP Error: (Request ID: Root=1-69b1eb80-71adc530557736b071097d90;d9f9dd39-1eb3-4ef0-ba3c-5262937b9357)

Bad request:
HTTP Error: (Request ID: Root=1-69b1eb8b-327d3b150fb21f54500eaa1b;53e79627-7379-4e6b-b427-8c7f26fab5c4)

Bad request:
HTTP Error: (Request ID: Root=1-69b1eb8b-327d3b150fb21f54500eaa1b;53e79627-7379-4e6b-b427-8c7f26fab5c4)

Bad request:
HTTP Error: (Request ID: Root=1-69b1eb96-03491cd562a1abed5f8cafe4;88b979cc-beaa-4cfd-bf6d-8c5d12e7cb32)

Bad request:
LLM suggested matches:
  1. ERROR
  2. ERROR
HTTP Error: (Request ID: Root=1-69b1eb96-03491cd562a1abed5f8cafe4;88b979cc-beaa-4cfd-bf6d-8c5d12e7cb32)

Bad request:
LLM suggested matches:
  1. ERROR
  2. ERROR


In [ ]:
BATCH_SIZE = 10
DELAY_BETWEEN_BATCHES = 5  

output_file = 'hf_llm_matches_progress.jsonl'

processed = set()
llm_matches = []
try:
    with open(output_file, 'r') as f:
        for line in f:
            entry = json.loads(line)
            llm_matches.append(entry)
            processed.add(entry['CompensationOffice1'])
    print(f"Resuming from {len(processed)} already processed entries")
except FileNotFoundError:
    print("Starting fresh")

df_remaining = no_matched_office_df[~no_matched_office_df['CompensationOffice1'].isin(processed)]
print(f"Remaining to process: {len(df_remaining)}")

total_batches = (len(df_remaining) + BATCH_SIZE - 1) // BATCH_SIZE
api_limit_reached = False

for batch_idx in tqdm(range(total_batches), desc="Processing batches"):
    if api_limit_reached:
        break
        
    start_idx = batch_idx * BATCH_SIZE
    end_idx = min(start_idx + BATCH_SIZE, len(df_remaining))
    
    batch_df = df_remaining.iloc[start_idx:end_idx]
    batch_names = batch_df['CompensationOffice1'].tolist()
    
    batch_results = get_llm_match_batch(batch_names, offices_list)
    
    if batch_results is None:
        print("\nAPI limit reached! Saving progress and stopping.")
        api_limit_reached = True
        break
    
    for i, (idx, row) in enumerate(batch_df.iterrows()):
        entry = {
            'CompensationOffice1': row['CompensationOffice1'],
            'MatchedOffice': row['MatchedOffice'],
            'MatchDistance': float(row['MatchDistance']) if pd.notna(row['MatchDistance']) else None,
            'LLMMatch': batch_results[i] if i < len(batch_results) else "ERROR"
        }
        llm_matches.append(entry)
        
        with open(output_file, 'a') as f:
            f.write(json.dumps(entry) + '\n')
    
    if batch_idx < total_batches - 1 and not api_limit_reached:
        time.sleep(DELAY_BETWEEN_BATCHES)

df_hf_matches = pd.DataFrame(llm_matches)
print(f"\nProcessed: {len(df_hf_matches)} entries")
if api_limit_reached:
    print("Note: Processing stopped due to API limit. Run again later to continue.")
df_hf_matches.head(20)

In [ ]:
valid_matches = df_hf_matches[~df_hf_matches['LLMMatch'].isin(['NO_MATCH', 'ERROR', 'PARSE_ERROR', 'RATE_LIMITED'])]
no_matches = df_hf_matches[df_hf_matches['LLMMatch'] == 'NO_MATCH']
errors = df_hf_matches[df_hf_matches['LLMMatch'].isin(['ERROR', 'PARSE_ERROR', 'RATE_LIMITED'])]

print(f"Total processed: {len(df_hf_matches)}")
print(f"LLM found matches: {len(valid_matches)} ({len(valid_matches)/max(len(df_hf_matches),1)*100:.1f}%)")
print(f"No match found: {len(no_matches)}")
print(f"Errors: {len(errors)}")

df_hf_matches.to_excel('compensation_office_hf_llm_matches.xlsx', index=False)
print("\nSaved to compensation_office_hf_llm_matches.xlsx")